## Test Grid SANOS



In [1]:
%load_ext autoreload
%autoreload 1

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"], reduced=True).data['SPX']
print(f"/done; columns {data.keys()}")



00: Importing torch...done; this took 1.1s.
00: Generating samples for 1 tickers SPX with from 2025-04-01 to 2025-07-01... done; this took 9ms.
00: Found the following number of samples per ticker: SPX: 63.
00: Found date range 2025-04-01 to 2025-07-01 with 63 total dates. First validation date for 10 steps per sample is 2025-06-04.
00: Using up to 1000 options per date.
01:   Ticker SPX: using 7 training samples (accross 44 training dates), and 2 validation samples (accross 19 validation dates).
00: Created 7 training samples and 2 validation samples based on the date split at 2025-06-04. That implies 22% of all samples are validation samples.
00: Total time taken: 0.014s.
/done; columns dict_keys(['dates', 'ask_vol', 'bid_vol', 'df', 'fit_vol', 'fwd', 'is_am', 'is_american', 'is_valid', 'log_open_interest', 'log_volume', 'mask_arb_free', 'nms', 'pure_ask_price', 'pure_bid_price', 'pure_fit_delta', 'pure_fit_dk', 'pure_fit_gamma', 'pure_fit_price', 'pure_fit_theta', 'pure_fit_vega', '

In [2]:
%aimport quantdata.utils.arbfree

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store, m_o_m
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.utils.arbfree import MkArbFree
from dataclasses import dataclass, field
import math
import numpy as np
import time

def fit(data):
    print(data.keys()   )
    num_samples = len(data.dates)

    mkArbFree = MkArbFree("surface", smoothness=0.01, mid_fit_weight=0.1 )

    s, o = mkArbFree.get_solver_and_solver_options(for_expiry=False)
    print(f"Solver: {s}, options: {o}")

    verbose = Context.all

    ticker = "SPX"
    for isample in range(num_samples-1,max(-1,num_samples-200),-20):
        d = data.dates[isample]
        verbose.write(f"{ticker} {d}:")        

        is_valid = data.is_valid[isample,:]
        uttys = np.unique(data.tte_y[isample,:][is_valid])
        r = PrettyObject()
        r.expiries = []
        r.strikes = []
        r.bids = []
        r.asks = []

        for utty in uttys:
            mask = (data.tte_y[isample,:] == utty) & is_valid
            r.expiries.append(utty)
            r.strikes.append(data.pure_strike[isample][mask])
            r.bids.append(data.pure_bid_price[isample][mask])
            r.asks.append(data.pure_ask_price[isample][mask])
            ixs = np.argsort(r.strikes[-1])
            r.strikes[-1] = r.strikes[-1][ixs]
            r.bids[-1] = r.bids[-1][ixs]
            r.asks[-1] = r.asks[-1][ixs]
        
        af = mkArbFree( expiries=r.expiries, strikes=r.strikes, bids=r.bids, asks=r.asks,
                        verbose=verbose(1) )





fit(data)


dict_keys(['dates', 'ask_vol', 'bid_vol', 'df', 'fit_vol', 'fwd', 'is_am', 'is_american', 'is_valid', 'log_open_interest', 'log_volume', 'mask_arb_free', 'nms', 'pure_ask_price', 'pure_bid_price', 'pure_fit_delta', 'pure_fit_dk', 'pure_fit_gamma', 'pure_fit_price', 'pure_fit_theta', 'pure_fit_vega', 'pure_strike', 'spot_ref', 'spot_ret', 'tte_y'])
Solver: HIGHS, options: {'highs_options': {'solver': 'simplex', 'presolve': 'on', 'primal_feasibility_tolerance': 1e-06, 'dual_feasibility_tolerance': 1e-06}, 'warm_start': True, 'time_limit': 300.0}
00: SPX 2025-07-01:
01:   Generated 47 0.01-smooth arbitrage-free expiries with extended strikes from 47 market expiries. This took 0.32s.
01:   Finding 0.01-smooth arbitrage-free mid prices accross the entire surface of 432 strikes with a minimum strike distance of 0.000806... done. This took 0.24s.
00: SPX 2025-06-02:
01:   Generated 43 0.01-smooth arbitrage-free expiries with extended strikes from 43 market expiries. This took 0.35s.
01:   Fin

01:   Generated 47 linear arbitrage-free expiries with extended strikes from 47 market expiries. This took 0.27s.
01:   Finding linear arbitrage-free mid prices accross the entire surface of 693 strikes with a minimum strike distance of 0.000806... done. This took 0.23s.
00: SPX 2025-06-02:
01:   Generated 43 linear arbitrage-free expiries with extended strikes from 43 market expiries. This took 0.21s.
01:   Finding linear arbitrage-free mid prices accross the entire surface of 684 strikes with a minimum strike distance of 0.000842... done. This took 0.41s.
00: SPX 2025-05-02:
01:   Generated 45 linear arbitrage-free expiries with extended strikes from 45 market expiries. This took 0.27s.
01:   Finding linear arbitrage-free mid prices accross the entire surface of 565 strikes with a minimum strike distance of 0.00088... done. This took 0.22s.
00: SPX 2025-04-03:
01:   Generated 46 linear arbitrage-free expiries with extended strikes from 46 market expiries. This took 0.35s.
01:   Finding linear arbitrage-free mid prices accross the entire surface of 406 strikes with a minimum strike distance of 0.000925... done. This took 0.16s.